# Recap - The Cars Dataset

In [ ]:
# 1 - DATA MANIPULATION
import pandas as pd
import numpy as np

# 2 - DATA VISUALISATION
import matplotlib.pyplot as plt
import seaborn as sns

# 3 - STATISTICS
from statsmodels.graphics.gofplots import qqplot

# 4 - MACHINE LEARNING

## 4.1 - Preprocessing

### 4.1.1 - Scalers
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

### 4.1.2 - Encoders
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

### 4.1.3 - Crossvalidation, Training, Model
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

### 4.1.4 - Evaluation
from sklearn.metrics import confusion_matrix
from sklearn.metrics  import ConfusionMatrixDisplay

# ?? The Cars dataset

?? Run the code down below to import the [***ML_Cars_Recap_dataset.csv dataset***](https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_Cars_Recap_dataset.csv) - _this version is slightly different from the one in the challenge 3._

In [ ]:
aws_url = "https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_Cars_Recap_dataset.csv"
cars = pd.read_csv(aws_url)
cars.head()


# Note: It is a common practice to store csv online. This one is stored on the AWS account of LeWagon

In [ ]:
cars.shape # 205 cars x 8 features

?? The description of the dataset is available [here](https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_Cars_dataset_description.txt).

? **Question (Information)** ? Collect some basic information about your dataset.

In [ ]:
cars['cylindernumber'] = cars['cylindernumber'].map({'four': 4,
 'six': 6,
 'five': 5,
 'eight': 8,
 'two': 2,
 'three': 3,
 'twelve': 12})

In [ ]:
cars['cylindernumber'].value_counts()

In [ ]:
cars.info()

# (1) ?? Data preparation

## (1.0) Pre-requisites

? **Question (Potential anomalies)** ? 

- Are there columns which need to be modified because of their data types? 
    - Please proceed accordingly.

In [ ]:
# YOUR CODE HERE

## (1.1) Duplicates

? **Question (Duplicated rows)** ? 

Compute the number of duplicated rows that the cars dataset has.

In [ ]:
cars.duplicated().sum()

In [ ]:
cars.drop_duplicates(inplace=True).reset_index()

? **Question (Remove duplicates)** ? 

In [ ]:
# YOUR CODE HERE

## (1.2) Missing data

? **Question (`NaN`)** ? 

What is the percentage of `NaN` for each feature?

In [ ]:
cars.isnull().sum()

<details>
    <summary><i>Answer</i></summary>
    
?? It is a miracle but we do not have missing values here. You can move to the next section of this notebook :) 
    
</details>

## (1.3) Defining the features $X$ and the target $y$

? **Question** ?

- Store the features of this dataset in a variable $X$
- Store the target in a variable $y$.

In [ ]:
cars.head()

In [ ]:
X = cars.drop(columns=['price'])
y = cars['price']

? **Question (Quick look at the target)** ? What is the proportion of "cheap" cars vs. "expensive" cars ? 

In [ ]:
# YOUR CODE HERE
(y == 'expensive').sum() / ((y == 'expensive').sum() + (y == 'cheap').sum())

In [ ]:
(y == 'cheap').sum() / ((y == 'expensive').sum() + (y == 'cheap').sum())

## (1.4) Scaling numerical features

? **Question (quick glance at the numerical columns)** ?

- Show the boxplots of all the numerical columns in one line of code
- Do the same with their distributions using histograms

In [ ]:
for c in cars.select_dtypes(include='number').columns:
    cars[[c]].boxplot()
    plt.show()

In [ ]:
# from statsmodels.graphics.gofplots import qqplot
for c in cars.select_dtypes(include='number').columns:
#     sns.histplot(cars[c],line='s')
    cars[[c]].hist()
    plt.show()

? **Question (Focus on the numerical columns)** ?

Store the numerical columns of the cars dataset into `cars_num`.

In [ ]:
###########################
# Manual solution         # - not very smart
###########################

# num_features = ["carwidth", "curbweight", "cylindernumber", "stroke", "peakrpm"]
# cars_num = X[num_features]

# That is one way to select the numerical features...
# But not very smart...

###########################
# Elegant solution        # - much better
###########################

cars_num = X.select_dtypes(exclude = ['object']) # selecting only the numerical features
cars_num

? **Question (Robust Scaler)** ?

??????? Robust Scaling all the numerical features is a quick and efficient way of scaling.

In [ ]:
cars

In [ ]:
# Instantiating a RobustScaler
robust_scaler = RobustScaler()

# Creating a copy of the original numerical features 
cars_num_scaled = cars_num.copy()

# Scaling the features and storing them in the copy
cars_num_scaled = pd.DataFrame(robust_scaler.fit_transform(cars_num_scaled),
                               columns = cars_num.columns)

# Showing the scaled features
cars_num_scaled

## (1.5) Encoding categorical features

? **Question (Focus on the categorical columns)** ?

Store the categorical columns of the cars dataset into `cars_categorical`.

In [ ]:
# Manual solution --> not very smart

# categorical_features = ["aspiration", "enginetype"]
# cars_categorical = X[categorical_features]

# Elegant solution --> much better
cars_categorical = X.select_dtypes(include = ['object']) # selecting only the categorical features
cars_categorical

In [ ]:
X.shape

? **Question: What is the expected number of columns generated if we encode the categorical features** ?

In [ ]:
cars['aspiration'].unique()

In [ ]:
cars['enginetype'].unique()

? **Question (OneHotEncoder): Encode the categorical features** ?

<details>
    <summary><i>Hints</i></summary>
    
- Drop a column when a categorical feature is binary
- Use full matrixes (Sklearn estimators don't understand sparse matrixes)
- Store them into a Pandas DataFrame called `encoded_features`
    - ?? Once that the OHE is fitted to a DataFrame, the new columns' names are accessible through _`get_features_names_out()`_
    
</details>

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

ohe = OneHotEncoder(sparse = False, drop="if_binary") 
# ohe.fit()
ohe.fit(X[['aspiration', 'enginetype']]) 

cars_categorical[ohe.get_feature_names_out()] = ohe.transform(X[['aspiration', 'enginetype']])

In [ ]:
cars_categorical.drop(columns=['aspiration', 'enginetype'], inplace=True)

In [ ]:
cars_categorical

In [ ]:
ohe.transform(X[['aspiration', 'enginetype']]).shape

## (1.6) Concatenating all the preprocessed features together $X_{preprocessed}$

? **Question (aggregating all the preprocessed features together)** ?

Concatenate `cars_num_scaled` and `cars_categorical` into `X_preprocessed`

In [ ]:
cars_num_scaled

In [ ]:
cars_categorical

In [ ]:
X_preprocessed = cars_num_scaled.join(cars_categorical)

In [ ]:
X_preprocessed['price'] = y

In [ ]:
X_preprocessed

In [ ]:
X_preprocessed.dropna(inplace=True)

In [ ]:
X_preprocessed.shape

## (1.7) Encoding the target

? **Question (Encoding the categorical target)** ?

Encode the target using ??[***LabelEncoder***](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).

In [ ]:
cars['price'].value_counts()

In [ ]:
y = X_preprocessed['price']

In [ ]:
X_preprocessed.drop(columns=['price'], inplace=True)

In [ ]:
X_preprocessed.head()

In [ ]:
y

# (2) ??????? Modeling (Logistic Regression)

?? Our goal is to predict whether a car will be expensive or cheap based on some features.

## (2.0) Cross-Validation

????? In Machine Learning, when you want to compare different models, a Cross Validation is a robust method to roughly assess the performance of each individual submodel.

? **Question (CrossValidation)** ?

What is the potential performance of a LogisticRegression for this dataset?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

model = LogisticRegression()
cv_results = cross_validate(model, X_preprocessed, y, cv=5)

In [ ]:
cv_results['test_score'].mean()

- ?? This accuracy is promising, let's go further !
- ?? We will discuss what is a good Regression/Classification model in the `Performance Metrics` unit.

## (2.1) Holdout Method

? **Question (Holdout): train/test split your dataset with a 70/30 ratio** ?

<details>
    <summary><i>Hint</i></summary>

- For a binary target, you actually don't have to use to `LabelEncoder`.
    - Sklearn is smart enough to detect a binary target.
    
</details>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.3, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

## (2.2) Training and evaluating

? **Question: Train/Evaluate a Logistic Regression model** ?

In [ ]:
# Instantiate a Logistic Regression model
logreg = LogisticRegression(max_iter = 100)

# Train the model
logreg.fit(X_train, y_train)

# Evaluate the model
logreg.score(X_test, y_test)

## (2.3) Predicting

? **Question (Predicting a new "point")**?

Does your ML algorithm (the Logistic Regression you've just trained) think that a car with the following features will be expensive? 

In [ ]:
len(['std', 65.88793969849246, 1874.0, 'ohc', 'four', 3.11, 5400.0])

In [ ]:
cars.columns

In [ ]:
robust_scaler.get_feature_names_out()

In [ ]:
new_car = pd.DataFrame(columns = ['aspiration', 'carwidth', 'curbweight', 'enginetype', 'cylindernumber',
       'stroke', 'peakrpm'])
new_car.loc[0] = ['std', 65.88793969849246, 1874.0, 'ohc', 'four', 3.11, 5400.0]
new_car

> ??????? **To make a prediction, this new car has to be through the same transformations as the ones performed in the training set!**

?? Have a look at the big picture of this recap [here](https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/prepare_dataset_fit_transform.png)!

In [ ]:
# 0 - Fixing abnormal data
pass  # YOUR CODE HERE

In [ ]:
new_car['cylindernumber'] = new_car['cylindernumber'].map({'four': 4,
 'six': 6,
 'five': 5,
 'eight': 8,
 'two': 2,
 'three': 3,
 'twelve': 12})

In [ ]:
robust_scaler.get_feature_names_out()

In [ ]:
# 1 - Scaling the numerical features
new_car[robust_scaler.get_feature_names_out()] = robust_scaler.transform(new_car[robust_scaler.get_feature_names_out()])

In [ ]:
new_car

In [ ]:
robust_scaler

In [ ]:
# 2 - Encoding the categorical features
new_car[ohe.get_feature_names_out()] = ohe.transform(new_car[['aspiration', 'enginetype']])   # YOUR CODE HERE

In [ ]:
new_car.drop(columns=['aspiration', 'enginetype'], inplace=True)

In [ ]:
# 3 - Aggregating the numerical features and categorical features together

new_car  # YOUR CODE HERE

In [ ]:
# 4 - Prediction
model.fit(X_preprocessed, y)
model.predict(new_car)

In [ ]:
model.coef_

In [ ]:
model.predict_proba(new_car)

# (3) ??????? Warning about Data Leakage

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/cross_validation_data_leakage.png"  width=600 height=300>

?? Congratulations! You are an expert at preparing a dataset now !

??????? Don't forget your flashcards and see you for the next section: `Performance Metrics`!